# Первичное исследование данных заказов Olist

## Цель анализа
Изучить структуру данных о заказах интернет-магазина Olist, проверить качество данных и выявить возможные несоответствия в датах и статусах заказов, которые необходимо учитывать при дальнейшем анализе сроков доставки.

## Источник данных
Brazilian E-Commerce Public Dataset by Olist (Kaggle).

## Содержание ноутбука

1. Первичное исследование данных.
2. Анализ распределения статусов заказов.
3. Проверка пропусков даты доставки по статусам заказов.
4. Проверка несоответствий между статусами заказов и датами доставки.
5. Преобразование столбцов с датами.
6. Проверка последовательности дат и выявление аномалий.
7. Итоги первичного исследования данных.

In [1]:
from pathlib import Path
import pandas as pd

## 1. Первичное исследование данных

На этом этапе изучим структуру таблицы заказов, типы данных и наличие пропущенных значений. Это позволит оценить качество данных и определить, какие проверки необходимо провести перед анализом сроков доставки.

In [2]:
DATA_DIR = Path("../data/raw")
orders = pd.read_csv(DATA_DIR / "olist_orders_dataset.csv")

print(f"Размер таблицы: {orders.shape}")
orders.head()

Размер таблицы: (99441, 8)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [4]:
print('Количество пропусков по столбцам:')
orders.isna().sum()

Количество пропусков по столбцам:


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

### Результаты первичного исследования

Исходная таблица содержит 99 441 запись и 8 столбцов.

На первом этапе были изучены структура таблицы, типы данных и наличие пропущенных значений. Это позволило получить общее представление о данных и определить поля, которые будут использоваться в дальнейшем анализе.

## 2. Анализ распределения статусов заказов

Перед исследованием сроков доставки необходимо изучить статусы заказов. Это поможет понять, какие заказы были доставлены, какие находятся в процессе обработки и какие не завершились доставкой.

In [5]:
status_distribution = pd.DataFrame({
    'Количество заказов': orders['order_status'].value_counts()
})

status_distribution['Доля, %'] = (
    status_distribution['Количество заказов'] / len(orders) * 100
).round(2)

status_distribution

,Количество заказов,"Доля, %"
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


## 3. Проверка пропусков даты доставки по статусам заказов

На этом этапе проверяется, как наличие пропущенной даты фактической доставки связано со статусом заказа. Это позволит выяснить, объясняются ли пропуски незавершёнными или отменёнными заказами, и выявить возможные несоответствия в данных.

In [6]:
orders.groupby('order_status').agg(
    total_orders=('order_id', 'count'),
    missing_delivery_date=('order_delivered_customer_date', lambda x: x.isna().sum()),
    available_delivery_date=('order_delivered_customer_date', lambda x: x.notna().sum())
)

,total_orders,missing_delivery_date,available_delivery_date
order_status,,,
approved,2,2,0
canceled,625,619,6
created,5,5,0
delivered,96478,8,96470
invoiced,314,314,0
processing,301,301,0
shipped,1107,1107,0
unavailable,609,609,0


### Результаты проверки пропусков даты доставки

У большинства заказов со статусом `delivered` дата фактической доставки заполнена. При этом у 8 заказов с данным статусом дата отсутствует.

Также у 6 заказов со статусом `canceled` указана дата доставки. Это сочетание статуса и даты требует дополнительной проверки.

У остальных статусов дата фактического получения покупателем отсутствует во всех записях. Это может объясняться тем, что заказы не были доставлены, однако само по себе отсутствие даты не позволяет однозначно установить причину.

## 4. Проверка несоответствий между статусами заказов и датами доставки

На предыдущем этапе обнаружены заказы, у которых статус не полностью согласуется с наличием даты фактической доставки. На этом этапе изучаются такие записи, чтобы оценить характер несоответствий и определить, требуют ли они дополнительной обработки перед дальнейшим анализом.

In [7]:
delivered_without_date = orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

delivered_without_date[[
    "order_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaN,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaN,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaN,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaN,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaN,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaN,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaN,2018-07-19 00:00:00


In [8]:
canceled_with_dates = orders[
    (orders["order_status"] == "canceled") &
    (orders["order_delivered_carrier_date"].notna()) &
    (orders["order_delivered_customer_date"].notna())
]

canceled_with_dates[[
    "order_id",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]]

,order_id,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
2921,1950d777989f6a877539f53795b4c3c3,2018-02-19 19:48:52,2018-02-19 20:56:05,2018-02-20 19:57:13,2018-03-21 22:03:51,2018-03-09 00:00:00
8791,dabf2b0e35b423f94618bf965fcb7514,2016-10-09 00:56:52,2016-10-09 13:36:58,2016-10-13 13:36:59,2016-10-16 14:36:59,2016-11-30 00:00:00
58266,770d331c84e5b214bd9dc70a10b829d0,2016-10-07 14:52:30,2016-10-07 15:07:10,2016-10-11 15:07:11,2016-10-14 15:07:11,2016-11-29 00:00:00
59332,8beb59392e21af5eb9547ae1a9938d06,2016-10-08 20:17:50,2016-10-09 14:34:30,2016-10-14 22:45:26,2016-10-19 18:47:43,2016-11-30 00:00:00
92636,65d1e226dfaeb8cdc42f665422522d14,2016-10-03 21:01:41,2016-10-04 10:18:57,2016-10-25 12:14:28,2016-11-08 10:58:34,2016-11-25 00:00:00
94399,2c45c33d2f9cb8ff8b1c86cc28c11c30,2016-10-09 15:39:56,2016-10-10 10:40:49,2016-10-14 10:40:50,2016-11-09 14:53:50,2016-12-08 00:00:00


### 4.1. Дополнительная проверка отменённых заказов

Для шести заказов со статусом `canceled`, у которых заполнены даты передачи перевозчику и фактического получения, дополнительно проверим наличие информации в таблицах товаров, оплат и отзывов.

In [9]:
items = pd.read_csv(DATA_DIR / "olist_order_items_dataset.csv")
payments = pd.read_csv(DATA_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(DATA_DIR / "olist_order_reviews_dataset.csv")

In [10]:
canceled_order_ids = canceled_with_dates["order_id"]

additional_info = pd.DataFrame({
    "order_id": canceled_order_ids.to_numpy(),
    "has_items": canceled_order_ids.isin(items["order_id"]).to_numpy(),
    "has_payments": canceled_order_ids.isin(payments["order_id"]).to_numpy(),
    "has_reviews": canceled_order_ids.isin(reviews["order_id"]).to_numpy()
})

additional_info

,order_id,has_items,has_payments,has_reviews
0,1950d777989f6a877539f53795b4c3c3,True,True,True
1,dabf2b0e35b423f94618bf965fcb7514,True,True,True
2,770d331c84e5b214bd9dc70a10b829d0,True,True,True
3,8beb59392e21af5eb9547ae1a9938d06,True,True,True
4,65d1e226dfaeb8cdc42f665422522d14,True,True,True
5,2c45c33d2f9cb8ff8b1c86cc28c11c30,True,True,True


По всем шести заказам найдены соответствующие записи во всех трёх таблицах. Для уточнения обстоятельств изучим оценки и текстовые комментарии покупателей.

In [11]:
canceled_reviews = reviews[
    reviews["order_id"].isin(canceled_order_ids)
]

canceled_reviews[[
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp"
]]

,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
10981,65d1e226dfaeb8cdc42f665422522d14,1,NaN,"O produto foi comprado em 10/10/2016, não cheg...",2016-11-09 00:00:00,2016-11-09 21:04:42
27200,2c45c33d2f9cb8ff8b1c86cc28c11c30,1,NaN,consta que meu pedido foi entregue mas nao che...,2016-11-10 00:00:00,2016-11-16 20:44:19
31748,770d331c84e5b214bd9dc70a10b829d0,1,NaN,"o produto é falso, ridiculo!",2016-12-01 00:00:00,2016-12-19 20:08:30
72219,dabf2b0e35b423f94618bf965fcb7514,5,NaN,Compraria novamente pq cumprem com o prazo de ...,2016-10-19 00:00:00,2016-10-19 20:43:13
92105,8beb59392e21af5eb9547ae1a9938d06,1,NaN,Produto não foi entregue. Não foi enviada nota...,2016-10-20 00:00:00,2016-10-24 17:18:09
96829,1950d777989f6a877539f53795b4c3c3,3,NaN,"O produto não foi enviado, porque eu errei ao ...",2018-03-11 00:00:00,2018-03-11 04:00:19


### Результаты проверки несоответствий

Среди заказов со статусом `delivered` обнаружено 8 записей без даты фактического получения. У 7 из них указана дата передачи заказа перевозчику, а у одного отсутствуют обе даты.

Также обнаружено 6 заказов со статусом `canceled`, у которых заполнены даты передачи перевозчику и фактического получения покупателем. В пяти случаях фактическая дата получения предшествовала плановой дате доставки, а в одном заказе доставка состоялась примерно на 13 дней позже плановой даты.

Для дополнительного изучения этих шести заказов были проверены таблицы товаров, оплат и отзывов. По всем шести заказам найдены соответствующие записи во всех трёх таблицах. В комментариях покупателей встречаются сообщения о недоставке и других проблемах с заказами, но также присутствуют положительные отзывы. Таким образом, дополнительные данные позволяют получить некоторый контекст, однако не дают однозначного объяснения несоответствий между статусами и датами.

Причины выявленных несоответствий по имеющимся данным установить нельзя. Записи не корректировались, поскольку недостаточно оснований для определения достоверных значений статусов и дат. При дальнейшем анализе учитывалось соответствие статуса заказа и заполненности дат.

## 5. Преобразование столбцов с датами

Для анализа сроков доставки необходимо преобразовать строковые значения в формат даты и времени. Это позволит сравнивать даты, выявлять некорректные последовательности событий и рассчитывать продолжительность доставки. Пропущенные значения при преобразовании останутся пропущенными.

In [12]:
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])

In [13]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

## 6. Проверка последовательности дат

Проверка проводится среди всех заказов с заполненными датами оформления, подтверждения, передачи перевозчику и получения покупателем, независимо от статуса заказа. 
Цель этапа — проверить хронологическую последовательность событий по заказам, для которых необходимые даты заполнены. Записи с пропущенными датами не включаются в соответствующие проверки, поскольку по ним невозможно проверить последовательность событий.

In [14]:
orders_with_all_dates = orders[orders['order_delivered_customer_date'].notna() 
                               & orders['order_purchase_timestamp'].notna()
                               & orders['order_delivered_carrier_date'].notna()
                               & orders['order_approved_at'].notna()
]

print('Получение раньше оформления заказа:', 
      sum(orders_with_all_dates['order_delivered_customer_date'] 
          < orders_with_all_dates['order_purchase_timestamp'])
)

print('Получение раньше подтверждения заказа:', 
      sum(orders_with_all_dates['order_delivered_customer_date'] 
          < orders_with_all_dates['order_approved_at'])
)

print('Получение раньше передачи перевозчику:', 
      sum(orders_with_all_dates['order_delivered_customer_date'] 
          < orders_with_all_dates['order_delivered_carrier_date'])
)

Получение раньше оформления заказа: 0
Получение раньше подтверждения заказа: 61
Получение раньше передачи перевозчику: 23


### 6.1. Проверка пересечения групп

Проверим, относятся ли заказы с нарушением последовательности между получением и передачей перевозчику к заказам с нарушением последовательности между получением и подтверждением.

In [15]:
condition_a = (
    orders_with_all_dates['order_delivered_customer_date']
    < orders_with_all_dates['order_delivered_carrier_date']
)

condition_b = (
    orders_with_all_dates['order_delivered_customer_date']
    < orders_with_all_dates['order_approved_at']
)

print('Пересечение групп:', (condition_a & condition_b).sum())

Пересечение групп: 0


Пересечение групп отсутствует: ни один заказ не имеет одновременно обоих выявленных нарушений хронологии. Таким образом, обнаружено 84 уникальных заказа с нарушенной последовательностью дат.

### 6.2. Заказы с получением раньше подтверждения

В 61 заказе дата фактического получения покупателем предшествует дате подтверждения заказа. Изучим эти записи подробнее, чтобы понять характер несоответствий.

In [16]:
orders_received_before_approval = orders_with_all_dates[
    orders_with_all_dates['order_delivered_customer_date']
    < orders_with_all_dates['order_approved_at']
]

approval_delivery_gap = (
    orders_received_before_approval['order_approved_at']
    - orders_received_before_approval['order_delivered_customer_date']
)

approval_delivery_gap.agg(['min', 'max', 'mean'])

min    0 days 00:05:19.000000
max    6 days 23:45:06.000000
mean   1 days 15:03:15.868852
dtype: timedelta64[us]

Минимальная разница между датой получения и подтверждения составляет около 5 минут, максимальная — почти 7 дней, средняя — около 1,6 дня. Эти результаты показывают, что величина расхождений неодинакова. Причины нарушения последовательности дат по имеющимся данным установить нельзя.

### 6.3. Заказы с получением раньше передачи перевозчику

В 23 заказах дата фактического получения покупателем предшествует дате передачи заказа перевозчику. Проверим величину этих расхождений.

In [17]:
orders_received_before_carrier = orders_with_all_dates[
    orders_with_all_dates['order_delivered_customer_date']
    < orders_with_all_dates['order_delivered_carrier_date']
]

carrier_delivery_gap = (
    orders_received_before_carrier['order_delivered_carrier_date']
    - orders_received_before_carrier['order_delivered_customer_date']
)

carrier_delivery_gap.agg(['min', 'max', 'mean'])

min     0 days 00:23:18.000000
max    16 days 02:18:29.000000
mean    3 days 06:27:18.478260
dtype: timedelta64[us]

Минимальная разница между датой получения покупателем и датой передачи заказа перевозчику составляет около 23 минут, максимальная — более 16 дней, средняя — около 3,27 дня. Размер расхождений существенно различается. Причины нарушения хронологии по имеющимся данным установить нельзя.

### 6.4. Итоги проверки последовательности дат

В ходе проверки обнаружены следующие несоответствия в хронологии событий:

- Не обнаружено заказов, у которых дата получения покупателем предшествует дате оформления.
- В 61 заказе дата получения предшествует дате подтверждения.
- В 23 заказах дата получения предшествует дате передачи перевозчику.
- Всего выявлено 84 уникальных заказа с одним из двух последних нарушений хронологии. Эти группы не пересекаются.

Причины несоответствий по имеющимся данным установить нельзя. При дальнейшем анализе сроков доставки необходимо учитывать качество дат и обоснованно выбирать записи для расчётов.

## 7. Итоги первичного исследования данных

В ходе первичного исследования изучены структура таблицы заказов, распределение статусов, пропуски и согласованность дат.

Обнаружены отдельные несоответствия между статусами заказов и датами доставки, а также 84 заказа с нарушениями хронологии между датой получения и датами подтверждения или передачи перевозчику. Причины этих несоответствий по имеющимся данным установить нельзя.

Для дальнейшего анализа сроков доставки будет использоваться отдельная выборка заказов со статусом `delivered` и заполненными фактической и плановой датами доставки. При интерпретации результатов важно учитывать, что предварительные проверки выявили особенности качества исходных данных.